# 🔬 Analyse Rapide - Indices Synthétiques Deriv

## ⚠️ IMPORTANT : Exécuter les cellules dans l'ordre !

**Instructions :**
1. Exécuter la cellule 1 (Imports) avec `Shift + Enter`
2. Modifier le SYMBOL dans la cellule 2 si besoin
3. Exécuter toutes les cellules suivantes avec `Shift + Enter`
4. Ou utiliser : Menu **Cell → Run All**

In [ ]:
# ========================================
# CELLULE 1 : IMPORTS (EXÉCUTER EN PREMIER !)
# ========================================

import sys
import os

# Ajouter le chemin parent
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Imports locaux
try:
    from src.analyzers.statistical_analyzer import StatisticalAnalyzer
    from src.analyzers.pattern_detector import PatternDetector
    print("✅ Imports réussis !")
except ImportError as e:
    print(f"❌ Erreur d'import : {e}")
    print("Vérifier que vous êtes dans le dossier notebooks/")

# Configuration
import warnings
warnings.filterwarnings('ignore')

print("\n✅ Configuration OK - Vous pouvez continuer !")

In [ ]:
# ========================================
# CELLULE 2 : CHARGEMENT DES DONNÉES
# ========================================

# ⚠️ MODIFIER ICI le symbole et le timeframe
SYMBOL = "Volatility_100_Index"  # Change selon tes données
TIMEFRAME = "M15"

# Charger les données
data_path = f"../data/raw/{SYMBOL}_{TIMEFRAME}.parquet"

print(f"Loading {SYMBOL} data from {data_path}...")

if not os.path.exists(data_path):
    print(f"\n❌ ERREUR : Fichier non trouvé : {data_path}")
    print("\nFichiers disponibles dans data/raw/ :")
    raw_dir = "../data/raw"
    if os.path.exists(raw_dir):
        files = [f for f in os.listdir(raw_dir) if f.endswith('.parquet')]
        for f in files:
            print(f"  - {f}")
    else:
        print("  Dossier data/raw/ n'existe pas !")
        print("  Exécutez d'abord : python src/extractors/deriv_api_extractor.py")
else:
    df = pd.read_parquet(data_path)
    
    print(f"\n✅ Data loaded: {len(df)} rows")
    print(f"📅 Period: {df.index[0]} to {df.index[-1]}")
    print(f"📊 Columns: {df.columns.tolist()}")
    print(f"\n🔍 First rows:")
    display(df.head())
    print(f"\n📈 Basic statistics:")
    display(df.describe())

In [ ]:
# ========================================
# CELLULE 3 : VISUALISATION DES PRIX
# ========================================

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Close'],
    mode='lines',
    name='Close Price',
    line=dict(color='blue', width=1)
))

fig.update_layout(
    title=f'{SYMBOL} - Close Price',
    xaxis_title='Time',
    yaxis_title='Price',
    hovermode='x unified',
    height=600
)

fig.show()

In [ ]:
# ========================================
# CELLULE 4 : CALCUL DES RENDEMENTS
# ========================================

returns = df['Close'].pct_change().dropna()

print(f"Returns calculated: {len(returns)} values")
print(f"Mean return: {returns.mean():.6f}")
print(f"Std return: {returns.std():.6f}")

# Plot
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Price', 'Returns'),
    vertical_spacing=0.1
)

fig.add_trace(
    go.Scatter(x=df.index, y=df['Close'], name='Price'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=returns.index, y=returns, name='Returns', line=dict(color='red')),
    row=2, col=1
)

fig.update_layout(height=800, showlegend=True)
fig.show()

In [ ]:
# ========================================
# CELLULE 5 : ANALYSE STATISTIQUE COMPLÈTE
# ========================================

print("Running statistical analysis...\n")

analyzer = StatisticalAnalyzer(df, price_column='Close')
results = analyzer.full_analysis()

print("✅ Analysis complete!\n")

# Afficher les résultats clés
print("=" * 60)
print("KEY RESULTS")
print("=" * 60)

# Distribution
dist = results['distribution']
print(f"\n📊 DISTRIBUTION:")
print(f"   Mean:     {dist['mean']:.6f}")
print(f"   Std:      {dist['std']:.6f}")
print(f"   Skewness: {dist['skewness']:.4f}")
print(f"   Kurtosis: {dist['kurtosis']:.4f}")

# Stationnarité
stat = results['stationarity']
print(f"\n🔄 STATIONARITY:")
print(f"   ADF test:  {'✅ Stationary' if stat['adf']['is_stationary'] else '❌ Non-stationary'} (p={stat['adf']['p_value']:.4f})")
print(f"   KPSS test: {'✅ Stationary' if stat['kpss']['is_stationary'] else '❌ Non-stationary'} (p={stat['kpss']['p_value']:.4f})")

# Hurst Exponent
hurst = results['hurst_exponent']
print(f"\n🎲 HURST EXPONENT:")
print(f"   Value: {hurst['hurst_exponent']:.4f}")
print(f"   {hurst['interpretation']}")

if hurst['hurst_exponent'] < 0.5:
    print("   ⚠️  MEAN REVERTING - Stratégies de retour à la moyenne !")
elif hurst['hurst_exponent'] > 0.5:
    print("   📈 TRENDING - Stratégies de suivi de tendance !")
else:
    print("   🎲 RANDOM WALK - Pas de pattern clair")

# Entropie
entropy = results['entropy']
print(f"\n🔐 ENTROPY (randomness):")
print(f"   Shannon:      {entropy['shannon_entropy']:.4f}")
print(f"   Sample:       {entropy['sample_entropy']:.4f}")
print(f"   Approximate:  {entropy['approximate_entropy']:.4f}")

# Cycles
freq = results['frequency']
if freq['dominant_periods']:
    print(f"\n🌊 DOMINANT CYCLES:")
    for i, period in enumerate(freq['dominant_periods'][:3]):
        print(f"   {i+1}. Period: {period:.1f} bars")

print("\n" + "=" * 60)

In [ ]:
# ========================================
# CELLULE 6 : DISTRIBUTION DES RENDEMENTS
# ========================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(returns, bins=100, alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution of Returns')
axes[0].set_xlabel('Returns')
axes[0].set_ylabel('Frequency')
axes[0].axvline(returns.mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()

# Q-Q plot
stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# CELLULE 7 : DÉTECTION DE PATTERNS
# ========================================

print("Detecting patterns...\n")

detector = PatternDetector(df, price_column='Close')
pattern_results = detector.full_pattern_analysis(spike_threshold=3.0)

print("✅ Pattern detection complete!\n")

# Afficher les résultats
spikes = pattern_results['spikes']
print("=" * 60)
print("PATTERN DETECTION RESULTS")
print("=" * 60)

print(f"\n⚡ SPIKES:")
print(f"   Crash spikes: {spikes['crash_spikes']['count']}")
print(f"   Boom spikes:  {spikes['boom_spikes']['count']}")

if spikes['crash_spikes']['count'] > 0:
    timing = spikes['crash_spikes']['timing']
    print(f"\n   Crash spike timing:")
    print(f"     Mean interval: {timing['mean_interval']:.1f} bars")
    print(f"     Std interval:  {timing['std_interval']:.1f} bars")

if spikes['boom_spikes']['count'] > 0:
    timing = spikes['boom_spikes']['timing']
    print(f"\n   Boom spike timing:")
    print(f"     Mean interval: {timing['mean_interval']:.1f} bars")
    print(f"     Std interval:  {timing['std_interval']:.1f} bars")

anomalies = pattern_results['anomalies']
print(f"\n🔍 ANOMALIES:")
print(f"   Detected: {anomalies['num_anomalies']}")
print(f"   Ratio:    {anomalies['anomaly_ratio']*100:.2f}%")

print("\n" + "=" * 60)

In [ ]:
# ========================================
# CELLULE 8 : VISUALISATION DES SPIKES
# ========================================

fig = go.Figure()

# Prix
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Close'],
    mode='lines',
    name='Price',
    line=dict(color='blue', width=1)
))

# Crash spikes
if spikes['crash_spikes']['count'] > 0:
    crash_indices = spikes['crash_spikes']['indices']
    crash_times = [df.index[i+1] for i in crash_indices if i+1 < len(df)]
    crash_prices = [df['Close'].iloc[i+1] for i in crash_indices if i+1 < len(df)]
    
    fig.add_trace(go.Scatter(
        x=crash_times,
        y=crash_prices,
        mode='markers',
        name='Crash Spikes',
        marker=dict(color='red', size=10, symbol='triangle-down')
    ))

# Boom spikes
if spikes['boom_spikes']['count'] > 0:
    boom_indices = spikes['boom_spikes']['indices']
    boom_times = [df.index[i+1] for i in boom_indices if i+1 < len(df)]
    boom_prices = [df['Close'].iloc[i+1] for i in boom_indices if i+1 < len(df)]
    
    fig.add_trace(go.Scatter(
        x=boom_times,
        y=boom_prices,
        mode='markers',
        name='Boom Spikes',
        marker=dict(color='green', size=10, symbol='triangle-up')
    ))

fig.update_layout(
    title='Spike Detection',
    xaxis_title='Time',
    yaxis_title='Price',
    height=600,
    hovermode='x unified'
)

fig.show()

In [ ]:
# ========================================
# CELLULE 9 : RÉSUMÉ FINAL
# ========================================

print("\n" + "=" * 70)
print(" " * 20 + "📊 ANALYSE SUMMARY 📊")
print("=" * 70)

print(f"\n📈 DATASET:")
print(f"   Symbol:      {SYMBOL}")
print(f"   Timeframe:   {TIMEFRAME}")
print(f"   Total bars:  {len(df)}")
print(f"   Period:      {df.index[0]} to {df.index[-1]}")

print(f"\n🎲 CARACTÈRE:")
print(f"   Hurst:       {hurst['hurst_exponent']:.4f} - {hurst['interpretation']}")
print(f"   Entropy:     {entropy['shannon_entropy']:.4f}")

if hurst['hurst_exponent'] < 0.4:
    print(f"\n💡 RECOMMANDATION: Mean Reversion Strategy")
    print(f"   - Acheter quand le prix est bas (oversold)")
    print(f"   - Vendre quand le prix est haut (overbought)")
    print(f"   - Utiliser RSI, Bollinger Bands")
elif hurst['hurst_exponent'] > 0.6:
    print(f"\n💡 RECOMMANDATION: Trend Following Strategy")
    print(f"   - Suivre les tendances établies")
    print(f"   - Utiliser moving averages, MACD")
    print(f"   - Trailing stop loss")
else:
    print(f"\n💡 RECOMMANDATION: Mixed Strategy")
    print(f"   - Comportement proche du random walk")
    print(f"   - Utiliser plusieurs indicateurs")
    print(f"   - Gestion du risque stricte")

print(f"\n⚡ PATTERNS DÉTECTÉS:")
print(f"   Crash spikes:  {spikes['crash_spikes']['count']}")
print(f"   Boom spikes:   {spikes['boom_spikes']['count']}")
print(f"   Anomalies:     {anomalies['num_anomalies']}")

if freq['dominant_periods']:
    print(f"   Cycles:        {freq['dominant_periods'][0]:.1f} bars (dominant)")

print("\n" + "=" * 70)
print("\n✅ Analyse terminée ! Vous pouvez maintenant :")
print("   1. Créer une stratégie basée sur ces résultats")
print("   2. Backtester la stratégie")
print("   3. Extraire d'autres indices pour comparer")
print("\n" + "=" * 70)